In [ ]:
from pathlib import Path
import sys
import matplotlib as mpl
CAMERA_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'figure_generation_notebooks' else Path.cwd().resolve()
EXPERIMENT_ROOT = Path('/mnt/e/watermark_rev2')
FIGURETYPE1_DIR = CAMERA_ROOT / 'figuretype1'
FIGURETYPE1_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EXPERIMENT_ROOT))
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import norm
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')
FONTSIZE_TITLE = 25
FONTSIZE_AXLABEL = 25
FONTSIZE_LEGEND = 18
FONTSIZE_TICK = 18
TICK_NBINS = 5
ASPECT_EQUAL = True
SCATTER_SIZE = 50
SCATTER_ALPHA = 0.4
SCATTER_ZORDER = 5
LINE_FIT_WIDTH = 4
LINE_FIT_ALPHA = 0.8
LINE_FIT_ZORDER = 4
LINE_REF_WIDTH = 2
LINE_REF_ALPHA = 0.8
LINE_REF_ZORDER = 8
PALETTE = 'muted'
COLOR_SCATTER = sns.color_palette(PALETTE)[0]
COLOR_FIT = sns.color_palette(PALETTE)[2]
COLOR_REF = 'gray'
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (6, 6), 'axes.spines.top': False, 'axes.spines.right': False})

def compute_ziid(green_token_mask, gamma, T, ngram_info=None):
    gt = np.array(green_token_mask)
    if ngram_info is not None:
        gt = gt[np.array(ngram_info) == True]
    if len(gt) < T:
        return None
    gt = gt[:T]
    S_T = np.sum(gt)
    denom = np.sqrt(T * gamma * (1 - gamma))
    return (S_T - T * gamma) / denom if denom > 0 else None

def create_qqplot(z_samples, title=None, ax=None):
    sns.set_theme(style='white', context='talk', font_scale=0.9)
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    M = len(z_samples)
    z_sorted = np.sort(z_samples)
    theoretical_quantiles = norm.ppf((np.arange(1, M + 1) - 0.5) / M)
    lims = [min(theoretical_quantiles.min(), z_sorted.min()) - 0.5, max(theoretical_quantiles.max(), z_sorted.max()) + 0.5]
    ax.plot(lims, lims, linestyle='--', linewidth=LINE_REF_WIDTH, color=COLOR_REF, alpha=LINE_REF_ALPHA, zorder=LINE_REF_ZORDER)
    z_mean = np.mean(z_samples)
    z_std = np.std(z_samples, ddof=1)
    fitted_line = z_mean + z_std * theoretical_quantiles
    ax.plot(theoretical_quantiles, fitted_line, linewidth=LINE_FIT_WIDTH, alpha=LINE_FIT_ALPHA, color=COLOR_FIT, label='Fitted', zorder=LINE_FIT_ZORDER)
    ax.scatter(theoretical_quantiles, z_sorted, s=SCATTER_SIZE, alpha=SCATTER_ALPHA, color=COLOR_SCATTER, edgecolor='none', label='Sample', zorder=SCATTER_ZORDER)
    ax.set_xlabel('Theoretical Quantiles N(0,1)', fontsize=FONTSIZE_AXLABEL)
    ax.set_ylabel(title + 'Sample Quantiles', fontsize=FONTSIZE_AXLABEL)
    ax.tick_params(labelsize=FONTSIZE_TICK)
    ax.locator_params(axis='both', nbins=TICK_NBINS)
    ax.grid(False)
    if ASPECT_EQUAL:
        ax.set_aspect('equal', adjustable='box')
    ax.legend(frameon=False, fontsize=FONTSIZE_LEGEND)
    sns.despine(ax=ax)
    return ax
results_dir = EXPERIMENT_ROOT / 'results'
qq_files = sorted([f for f in results_dir.glob('QQ*.json') if not f.name.endswith('_config.json') and 'd0.00000' in f.name])
T_values = [50, 100, 150]
h0_samples = []
h0_config = None
for file_path in qq_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if h0_config is None:
        h0_config = {'gamma': data['config']['gamma'], 'delta': data['config']['delta']}
    for result in data['results']:
        gt = result['watermarked']['green_token_mask']
        ngram_info = result['watermarked'].get('ngram_info', None)
        h0_samples.append({'green_token_mask': np.array(gt), 'ngram_info': np.array(ngram_info) if ngram_info else None})
h1_files = sorted([f for f in results_dir.glob('QQ*.json') if not f.name.endswith('_config.json') and 'd1.00000' in f.name])
h1_samples = []
h1_config = None
for file_path in h1_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if h1_config is None:
        h1_config = {'gamma': data['config']['gamma'], 'delta': data['config']['delta']}
    for result in data['results']:
        gt = result['watermarked']['green_token_mask']
        ngram_info = result['watermarked'].get('ngram_info', None)
        h1_samples.append({'green_token_mask': np.array(gt), 'ngram_info': np.array(ngram_info) if ngram_info else None})
h0_gamma_emp_by_T = {}
h0_ziid_by_T = {}
for T in T_values:
    green_rates = []
    for sample in h0_samples:
        gt = np.array(sample['green_token_mask'])
        ng = sample['ngram_info']
        if ng is not None:
            gt = gt[np.array(ng) == True]
        if len(gt) >= T:
            green_rates.append(np.mean(gt[:T]))
    if not green_rates:
        continue
    gamma_emp = np.mean(green_rates)
    h0_gamma_emp_by_T[T] = gamma_emp
    z_values = [compute_ziid(s['green_token_mask'], gamma_emp, T, s['ngram_info']) for s in h0_samples]
    z_values = [z for z in z_values if z is not None]
    if z_values:
        h0_ziid_by_T[T] = np.array(z_values)
h1_gamma_emp_by_T = {}
h1_ziid_by_T = {}
for T in T_values:
    green_rates = []
    for sample in h1_samples:
        gt = np.array(sample['green_token_mask'])
        ng = sample['ngram_info']
        if ng is not None:
            gt = gt[np.array(ng) == True]
        if len(gt) >= T:
            green_rates.append(np.mean(gt[:T]))
    if not green_rates:
        continue
    gamma_emp = np.mean(green_rates)
    h1_gamma_emp_by_T[T] = gamma_emp
    z_values = [compute_ziid(s['green_token_mask'], gamma_emp, T, s['ngram_info']) for s in h1_samples]
    z_values = [z for z in z_values if z is not None]
    if z_values:
        h1_ziid_by_T[T] = np.array(z_values)
FIGURETYPE1_DIR.mkdir(parents=True, exist_ok=True)
for T in T_values:
    if T not in h0_ziid_by_T or T not in h1_ziid_by_T:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
    create_qqplot(h0_ziid_by_T[T], title='$H_0$ ', ax=axes[0])
    create_qqplot(h1_ziid_by_T[T], title='$H_1$ ', ax=axes[1])
    plt.tight_layout()
    plt.savefig(FIGURETYPE1_DIR / f'qqplot_H0_H1_side_by_side_T{T}.pdf', dpi=300, bbox_inches='tight')
    plt.show()
